# 🔄 Notebook 7: Server-Side Push/Pull

We've covered how to get updates from server to client (Hop 1). Now let's explore **Hop 2**: how updates get from the **source** to the **server**!

## Learning Objectives

By the end of this notebook, you'll understand:
- The "Two Hops" problem in real-time systems
- Pull-based approach (polling from database)
- Push via consistent hashing
- Push via Pub/Sub (Redis)
- When to use each approach

---

### 🔍 Open RedisInsight to Watch Pub/Sub in Action

1. Go to **http://localhost:5540**
2. Click "Add Redis Database"
3. Enter: Host = `redis`, Port = `6379`, Name = `realtime-redis`
4. Click "Add Redis Database"
5. Open the "Pub/Sub" tab to see messages flowing!

💡 **Tip**: Watch for messages on channels like `chat:general` when you run the Pub/Sub demos!

## 🔄 The Two Hops Problem

In a real-time system, updates need to travel two hops:

```
┌──────────┐     Hop 2      ┌──────────┐     Hop 1      ┌──────────┐
│  Source  │ ─────────────► │  Server  │ ─────────────► │  Client  │
│ (Events) │                │          │                │          │
└──────────┘                └──────────┘                └──────────┘

Examples of sources:
• User A sends a message → Server → User B's browser
• Driver updates location → Server → Passenger's phone
• Stock price changes → Server → Trader's dashboard
```

We've covered Hop 1 (Polling, Long Polling, SSE, WebSocket, WebRTC).

Now let's focus on **Hop 2**: How does the server know there's a new update?

## 📂 Approach 1: Pull via Database Polling

The simplest approach: store updates in a database, let clients poll for them.

```
┌────────┐     writes      ┌──────────┐     polls     ┌────────┐
│ User A │ ───────────────►│ Database │◄──────────────│ User B │
│(source)│                 │          │               │(client)│
└────────┘                 └──────────┘               └────────┘
```

The database acts as a **buffer** between the source and the consumer.

In [ ]:
# Example: Database-backed polling

from datetime import datetime
import time
import threading

# Simulate a database with a simple list
class SimpleDatabase:
    def __init__(self):
        self.messages = []
        self.lock = threading.Lock()
    
    def insert(self, message):
        """Insert a new message (source writes here)"""
        with self.lock:
            msg = {
                "id": len(self.messages) + 1,
                "text": message,
                "timestamp": datetime.now().timestamp()
            }
            self.messages.append(msg)
            return msg
    
    def query_since(self, since_timestamp):
        """Query messages after a timestamp (client polls here)"""
        with self.lock:
            return [m for m in self.messages if m["timestamp"] > since_timestamp]

# Create our "database"
db = SimpleDatabase()

print("📂 Database-backed Polling Example")
print("="*50)

# Source writes messages
def source_writes():
    for i in range(3):
        time.sleep(1)
        msg = db.insert(f"Message {i+1}")
        print(f"📤 Source wrote: {msg['text']}")

# Client polls for messages
def client_polls():
    last_timestamp = datetime.now().timestamp()
    for _ in range(5):
        time.sleep(0.8)
        new_messages = db.query_since(last_timestamp)
        if new_messages:
            for m in new_messages:
                print(f"📬 Client received: {m['text']}")
                last_timestamp = max(last_timestamp, m['timestamp'])
        else:
            print("📭 Client poll: nothing new")

# Run both
source_thread = threading.Thread(target=source_writes)
client_thread = threading.Thread(target=client_polls)

source_thread.start()
client_thread.start()

source_thread.join()
client_thread.join()

print("\n✅ Database decouples source from client!")

### Pros & Cons of Database Polling

**Pros:**
- 🟢 Very simple to implement
- 🟢 Source and client are completely decoupled
- 🟢 Messages are persisted (can replay)
- 🟢 Stateless servers

**Cons:**
- 🔴 Not real-time (latency = poll interval)
- 🔴 High DB load with frequent polling
- 🔴 Most polls return empty

## 🎯 Approach 2: Push via Consistent Hashing

For real-time updates with WebSocket/SSE, we need to PUSH to the right server. But which server holds User B's connection?

**Solution: Hash the user ID to determine the server!**

```
hash("userB") % num_servers = server_index
```

In [ ]:
# Demonstrate simple hashing for server assignment

def simple_hash_assignment(user_id: str, num_servers: int) -> int:
    """
    Determine which server handles a user using simple hashing.
    """
    hash_value = hash(user_id)
    return hash_value % num_servers

print("🎯 Simple Hash-based Server Assignment")
print("="*50)

users = ["alice", "bob", "charlie", "david", "eve"]
num_servers = 3

print(f"\nWith {num_servers} servers:\n")
for user in users:
    server = simple_hash_assignment(user, num_servers)
    print(f"  {user:10} → Server {server}")

print("\n" + "="*50)
print("\n⚠️ Problem: What happens when we add a 4th server?\n")

num_servers = 4
print(f"With {num_servers} servers:\n")
for user in users:
    server = simple_hash_assignment(user, num_servers)
    print(f"  {user:10} → Server {server}")

print("\n😱 Almost ALL users moved to different servers!")
print("   This means ALL connections need to reconnect!")

### Consistent Hashing to the Rescue!

**Consistent hashing** minimizes reassignments when servers are added/removed.

In [ ]:
# Implement consistent hashing

import hashlib
from bisect import bisect_left

class ConsistentHashRing:
    """
    Consistent hash ring for server assignment.
    """
    
    def __init__(self, replicas: int = 100):
        self.replicas = replicas  # Virtual nodes per server
        self.ring = []  # Sorted list of (hash, server)
        self.servers = set()
    
    def _hash(self, key: str) -> int:
        """Hash a key to a position on the ring."""
        return int(hashlib.md5(key.encode()).hexdigest(), 16)
    
    def add_server(self, server: str):
        """Add a server to the ring."""
        self.servers.add(server)
        for i in range(self.replicas):
            key = f"{server}:{i}"
            hash_val = self._hash(key)
            self.ring.append((hash_val, server))
        self.ring.sort()
    
    def remove_server(self, server: str):
        """Remove a server from the ring."""
        self.servers.discard(server)
        self.ring = [(h, s) for h, s in self.ring if s != server]
    
    def get_server(self, key: str) -> str:
        """Get the server for a given key."""
        if not self.ring:
            return None
        
        hash_val = self._hash(key)
        
        # Find the first server with hash >= key's hash
        hashes = [h for h, s in self.ring]
        idx = bisect_left(hashes, hash_val)
        
        # Wrap around to first server if past the end
        if idx == len(self.ring):
            idx = 0
        
        return self.ring[idx][1]

# Demo consistent hashing
print("🎯 Consistent Hashing Demo")
print("="*50)

ring = ConsistentHashRing(replicas=50)

# Add 3 servers
for server in ["server-1", "server-2", "server-3"]:
    ring.add_server(server)

users = ["alice", "bob", "charlie", "david", "eve", "frank", "grace"]

print("\nWith 3 servers:")
assignments_before = {}
for user in users:
    server = ring.get_server(user)
    assignments_before[user] = server
    print(f"  {user:10} → {server}")

# Add a 4th server
print("\n➕ Adding server-4...\n")
ring.add_server("server-4")

print("With 4 servers:")
moved = 0
for user in users:
    server = ring.get_server(user)
    status = "" if server == assignments_before[user] else " (MOVED)"
    if status:
        moved += 1
    print(f"  {user:10} → {server}{status}")

print(f"\n✅ Only {moved}/{len(users)} users moved!")
print("   Much better than simple hashing where almost all moved!")

### How Consistent Hashing Enables Push

```
1. User B connects:
   hash("userB") → Server 2
   Server 2 stores the WebSocket connection

2. User A sends message to User B:
   API Server: hash("userB") → Server 2
   API Server: Send message to Server 2
   Server 2: Push to User B's WebSocket
```

In [ ]:
# Visualize the consistent hashing architecture

print("🏗️ Consistent Hashing Architecture")
print("="*60)
print("""
                    ┌─────────────────────┐
                    │   Coordination      │
                    │   (ZooKeeper/etcd)  │
                    │                     │
                    │ Stores: N servers,  │
                    │ their addresses     │
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           │                   │                   │
           ▼                   ▼                   ▼
    ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
    │  WS Server  │     │  WS Server  │     │  WS Server  │
    │      1      │     │      2      │     │      3      │
    └──────┬──────┘     └──────┬──────┘     └──────┬──────┘
           │                   │                   │
           │                   │                   │
    [Alice, Eve]         [Bob, David]       [Charlie]
    (connections)        (connections)      (connections)

Message Flow:
─────────────
1. Alice sends message to Bob
2. API Server: hash("bob") = Server 2
3. API Server → Server 2: "deliver to bob"
4. Server 2 → Bob's WebSocket: message!
""")

### Pros & Cons of Consistent Hashing

**Pros:**
- 🟢 Predictable server assignment
- 🟢 Minimal disruption when scaling
- 🟢 Direct routing to correct server

**Cons:**
- 🔴 Requires coordination service
- 🔴 Complex deployment/scaling logic
- 🔴 All servers need routing table
- 🔴 Connection state is lost on server failure

## 📨 Approach 3: Push via Pub/Sub

The most flexible approach: use a **Pub/Sub system** (like Redis) to broadcast updates!

```
┌────────┐     publish     ┌──────────┐     subscribe     ┌────────┐
│ Source │ ───────────────►│  Redis   │◄─────────────────│ Server │
│        │                 │  Pub/Sub │                   │        │
└────────┘                 └──────────┘                   └────────┘
```

---

### 🔍 Watch Pub/Sub in RedisInsight!

1. Open **http://localhost:5540**
2. Go to your database → **Pub/Sub** tab
3. Click **Subscribe** and enter `chat:*` to watch all chat channels
4. Run the cells below and watch messages flow in real-time!

In [ ]:
# Simulate Pub/Sub (without actual Redis)

import threading
from collections import defaultdict
from queue import Queue
import time

class SimplePubSub:
    """
    A simple in-memory Pub/Sub system.
    In production, use Redis Pub/Sub!
    """
    
    def __init__(self):
        self.subscribers = defaultdict(list)  # channel -> [queues]
        self.lock = threading.Lock()
    
    def subscribe(self, channel: str) -> Queue:
        """Subscribe to a channel. Returns a queue for receiving messages."""
        queue = Queue()
        with self.lock:
            self.subscribers[channel].append(queue)
        return queue
    
    def unsubscribe(self, channel: str, queue: Queue):
        """Unsubscribe from a channel."""
        with self.lock:
            if queue in self.subscribers[channel]:
                self.subscribers[channel].remove(queue)
    
    def publish(self, channel: str, message: dict):
        """Publish a message to all subscribers of a channel."""
        with self.lock:
            for queue in self.subscribers[channel]:
                queue.put(message)
        return len(self.subscribers[channel])

# Create our pub/sub system
pubsub = SimplePubSub()

print("📨 Pub/Sub Demo")
print("="*50)

In [ ]:
# Demo: Chat room with Pub/Sub

def simulate_ws_server(server_id: str, users: list, duration: float = 5):
    """
    Simulate a WebSocket server that subscribes to user channels.
    """
    print(f"🖥️ Server {server_id} starting with users: {users}")
    
    # Subscribe to each user's channel
    queues = {}
    for user in users:
        channel = f"user:{user}"
        queues[user] = pubsub.subscribe(channel)
        print(f"   └─ Subscribed to {channel}")
    
    # Listen for messages
    start = time.time()
    while time.time() - start < duration:
        for user, queue in queues.items():
            if not queue.empty():
                msg = queue.get_nowait()
                print(f"📬 Server {server_id}: Delivering to {user}: {msg}")
        time.sleep(0.1)
    
    # Cleanup
    for user in users:
        pubsub.unsubscribe(f"user:{user}", queues[user])
    
    print(f"🖥️ Server {server_id} stopping")

def send_message(from_user: str, to_user: str, text: str):
    """
    Send a message to a user via Pub/Sub.
    """
    channel = f"user:{to_user}"
    message = {
        "from": from_user,
        "text": text,
        "timestamp": time.time()
    }
    count = pubsub.publish(channel, message)
    print(f"📤 {from_user} → {to_user}: '{text}' (delivered to {count} subscriber(s))")

# Start two servers with different users
server1_thread = threading.Thread(
    target=simulate_ws_server,
    args=("1", ["alice", "bob"], 4)
)
server2_thread = threading.Thread(
    target=simulate_ws_server,
    args=("2", ["charlie"], 4)
)

print("\n🚀 Starting servers...\n")
server1_thread.start()
server2_thread.start()

time.sleep(1)  # Let servers start

# Send some messages
print("\n📨 Sending messages...\n")
send_message("charlie", "alice", "Hello Alice!")
time.sleep(0.5)
send_message("alice", "bob", "Hey Bob!")
time.sleep(0.5)
send_message("bob", "charlie", "Hi Charlie!")

# Wait for servers to finish
server1_thread.join()
server2_thread.join()

print("\n✅ Demo complete!")
print("\n💡 Key insight: Servers don't need to know about each other!")
print("   Pub/Sub handles message routing automatically.")

### 🔴 Real Redis Pub/Sub Demo

Now let's use **actual Redis** so you can see messages in RedisInsight!

In [ ]:
import redis
import json
import threading
import time

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

try:
    r.ping()
    print("✅ Connected to Redis")
except:
    print("❌ Redis not running. Start with: docker-compose up -d")

def redis_subscriber(channel_pattern: str, duration: float = 5):
    pubsub = r.pubsub()
    pubsub.psubscribe(channel_pattern)
    print(f"📡 Subscribed to: {channel_pattern}")
    
    start = time.time()
    for message in pubsub.listen():
        if time.time() - start > duration:
            break
        if message['type'] == 'pmessage':
            data = json.loads(message['data'])
            print(f"📬 Received on {message['channel']}: {data}")
    
    pubsub.punsubscribe(channel_pattern)
    print("📡 Unsubscribed")

subscriber_thread = threading.Thread(
    target=redis_subscriber,
    args=("chat:*", 6)
)
subscriber_thread.start()
time.sleep(0.5)

print("\n📤 Publishing messages to Redis...")
print("   👀 Watch RedisInsight Pub/Sub tab!\n")

r.publish("chat:general", json.dumps({"from": "alice", "text": "Hello everyone!"}))
time.sleep(1)
r.publish("chat:general", json.dumps({"from": "bob", "text": "Hey Alice!"}))
time.sleep(1)
r.publish("chat:random", json.dumps({"from": "charlie", "text": "Anyone here?"}))

subscriber_thread.join()
print("\n✅ Real Redis Pub/Sub demo complete!")

### Pub/Sub Architecture

In [ ]:
# Visualize Pub/Sub architecture

print("🏗️ Pub/Sub Architecture")
print("="*60)
print("""
                    ┌─────────────────────┐
                    │   Load Balancer     │
                    │  (Least Connections)│
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           │                   │                   │
           ▼                   ▼                   ▼
    ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
    │  Endpoint   │     │  Endpoint   │     │  Endpoint   │
    │  Server 1   │     │  Server 2   │     │  Server 3   │
    └──────┬──────┘     └──────┬──────┘     └──────┬──────┘
           │                   │                   │
           │ subscribe         │ subscribe         │ subscribe
           │                   │                   │
           └───────────────────┼───────────────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │      Redis          │
                    │     Pub/Sub         │
                    └──────────┬──────────┘
                               │
                               │ publish
                               │
                    ┌──────────┴──────────┐
                    │   Message Sender    │
                    │   (API/Backend)     │
                    └─────────────────────┘

Flow:
─────
1. Client connects to ANY endpoint server
2. Endpoint server subscribes to user's channel on Redis
3. When message needs to be sent:
   - Backend publishes to Redis channel
   - Redis broadcasts to all subscribers
   - The right endpoint server receives it
   - Endpoint server pushes to client's WebSocket
""")

### Pros & Cons of Pub/Sub

**Pros:**
- 🟢 Endpoint servers are stateless (easy to scale)
- 🟢 No need for coordination service
- 🟢 Any server can handle any user
- 🟢 Simple load balancing

**Cons:**
- 🔴 Redis is single point of failure
- 🔴 Extra hop through Pub/Sub adds latency (~10ms)
- 🔴 Doesn't know if user is actually connected
- 🔴 Many-to-many connections between servers and Pub/Sub

## 📊 Comparison: When to Use Each Approach

In [ ]:
# Comparison table

print("📊 Server-Side Push/Pull Comparison")
print("="*70)
print("""
                    DB Polling       Consistent Hash      Pub/Sub
────────────────────────────────────────────────────────────────────
Real-time?          ❌ No            ✅ Yes               ✅ Yes
Complexity          🟢 Low           🔴 High              🟡 Medium
Stateless servers   ✅ Yes           ❌ No (connections)  ✅ Yes
Scaling             🟢 Easy          🟡 Complex           🟢 Easy
Extra latency       Poll interval    ~0ms                 ~10ms
Single point fail   DB               Coord service        Pub/Sub
Best for            Non-realtime     Heavy state          General use

────────────────────────────────────────────────────────────────────

DECISION GUIDE:
───────────────
• Need real-time? 
  No  → Database Polling ✅
  Yes → Continue...

• Have heavy per-connection state?
  Yes → Consistent Hashing ✅
  No  → Pub/Sub ✅

• Examples:
  - Email notifications → Database Polling
  - Google Docs (document state) → Consistent Hashing  
  - Chat application → Pub/Sub
""")

## 🎯 Interview Tips

In [ ]:
# Interview talking points

print("🎯 Interview Talking Points")
print("="*60)
print("""
THE TWO HOPS FRAMEWORK:
"When designing a real-time system, I think about two hops:
 1. How do updates get from the source to my server?
 2. How do updates get from my server to the client?"

FOR DATABASE POLLING:
"If updates don't need to be truly real-time, I'd use database
 polling. It's simple, and the DB naturally decouples producers
 from consumers. The trade-off is latency equals poll interval."

FOR CONSISTENT HASHING:
"For heavy state like collaborative editing, I'd use consistent
 hashing to route users to specific servers. This keeps document
 state local. I'd use ZooKeeper/etcd for coordination and minimize
 reconnection disruption during scaling."

FOR PUB/SUB:
"For most real-time apps like chat, I prefer Pub/Sub with Redis.
 Endpoint servers stay stateless - they just hold connections and
 subscribe to channels. When a message needs delivery, we publish
 to the user's channel and the right server receives it."

SCALING PUB/SUB:
"Redis Pub/Sub can become a bottleneck. I'd use Redis Cluster to
 shard channels across nodes. For very high scale, Kafka provides
 better durability and throughput."
""")

## 🧪 Quick Quiz

1. **You're building a notification system that needs to deliver alerts within 5 seconds. Database polling or Pub/Sub?**

2. **Why is consistent hashing better for Google Docs than Pub/Sub?**

3. **What's the main advantage of Pub/Sub over consistent hashing for a chat app?**

In [ ]:
# Quiz answers

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. DATABASE POLLING would work!")
print("   5 seconds is long enough that you don't need")
print("   real-time infrastructure. Poll every 2-3 seconds.")
print("")
print("2. GOOGLE DOCS has heavy per-document state!")
print("   Each document needs to track operations, cursor")
print("   positions, and conflict resolution state.")
print("   Consistent hashing keeps this state on ONE server.")
print("")
print("3. STATELESS SERVERS!")
print("   With Pub/Sub, users can connect to ANY server.")
print("   Scaling is simple: just add servers.")
print("   No coordination service needed.")

## 📚 Summary

### What We Learned:

1. **Two Hops** - Source→Server (Hop 2), Server→Client (Hop 1)

2. **Database Polling**
   - Simple, not real-time
   - Good for non-urgent updates

3. **Consistent Hashing**
   - Route users to specific servers
   - Good for heavy per-connection state
   - Complex scaling

4. **Pub/Sub (Redis)**
   - Stateless endpoint servers
   - Easy scaling
   - Good for most real-time apps

### The Complete Picture:

```
┌────────┐     Hop 2        ┌────────┐     Hop 1        ┌────────┐
│ Source │ ───────────────► │ Server │ ───────────────► │ Client │
└────────┘                  └────────┘                  └────────┘
              │                              │
              │                              │
     ┌────────┴────────┐          ┌─────────┴─────────┐
     │ DB Polling      │          │ Simple Polling    │
     │ Consistent Hash │          │ Long Polling      │
     │ Pub/Sub         │          │ SSE               │
     └─────────────────┘          │ WebSocket         │
                                  │ WebRTC            │
                                  └───────────────────┘
```

### 🎉 Congratulations!

You've completed the Real-time Updates pattern series! You now understand:
- All client-server protocols (Polling → WebRTC)
- Server-side update propagation strategies
- When to use each approach
- How to discuss these in interviews!